# Required Models From Scratch — CART, Logistic Regression, SVM

Notebook ini menjelaskan ketiga implementasi manual yang diwajibkan. Kode inti berada di `src/dtl_lr_svm/`.

## 1. CART Decision Tree

CART memilih binary split yang menurunkan weighted Gini impurity paling besar:

$$
\operatorname{Gini}(S) = 1 - \sum_k p_k^2
$$

$$
\Delta \operatorname{Gini} = \operatorname{Gini}(S) - \frac{n_L}{n}\operatorname{Gini}(S_L) - \frac{n_R}{n}\operatorname{Gini}(S_R)
$$

Pseudocode:

```text
build(node, depth):
    if stopping_condition: return leaf
    for each feature:
        evaluate valid threshold candidates
        compute weighted Gini decrease
    choose best split
    recursively build left and right child
```

Approximate candidate caps (`max_thresholds_per_feature`) dipakai untuk mengontrol biaya pencarian split.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT))

from src.dtl_lr_svm.cart_scratch import CARTClassifierScratch, CARTConfig

cart = CARTClassifierScratch(
    CARTConfig(
        max_depth=8,
        min_samples_split=40,
        min_samples_leaf=20,
        class_weight="balanced",
        max_thresholds_per_feature=128,
        random_state=42,
    )
)
print(cart.config)

## 2. Logistic Regression

$$
\hat{p}_i = \sigma\!\left(\mathbf{w}^{\top}\mathbf{x}_i + b\right), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}
$$

Weighted BCE + L2:

$$
J(\mathbf{w}, b) = -\frac{1}{N}\sum_i \omega_i \left[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)\right] + \frac{\lambda}{2}\lVert\mathbf{w}\rVert_2^2
$$

Gradient descent memperbarui parameter sampai max epoch atau early stopping.

In [ ]:
from src.dtl_lr_svm.logistic_regression_scratch import (
    LogisticRegressionScratch,
    LogisticRegressionConfig,
)

lr = LogisticRegressionScratch(
    LogisticRegressionConfig(
        learning_rate=0.05,
        l2=1e-3,
        class_weight="balanced",
        random_state=42,
    )
)
print(lr.config)

## 3. Linear SVM

Primal objective:

$$
J(\mathbf{w}, b) = \frac{\lambda}{2}\lVert\mathbf{w}\rVert_2^2 + \frac{1}{N}\sum_i \omega_i \max\!\left(0, 1-y_i(\mathbf{w}^{\top}\mathbf{x}_i+b)\right)
$$

Untuk sampel dengan margin `< 1`, subgradient hinge loss aktif. Model dilatih manual dengan batch/mini-batch subgradient descent dan threshold decision score dituning pada validation data.

In [ ]:
from src.dtl_lr_svm.svm_scratch import LinearSVMScratch, LinearSVMConfig

svm = LinearSVMScratch(
    LinearSVMConfig(
        regularization=1e-4,
        learning_rate=0.01,
        class_weight="balanced",
        random_state=42,
    )
)
print(svm.config)

## Ringkasan hasil utama

- CART scratch: OOF macro F1 ≈ **0.87198**, public Kaggle **0.85406**.
- Nonlinear-preprocessed LR: OOF ≈ **0.86435**.
- Nonlinear/squared-hinge SVM: OOF ≈ **0.86482**.

CART menjadi baseline manual terkuat sebelum external-data optimization.